In [1]:
# train_distilmbert_resume.py
# pip install -U torch transformers scikit-learn pandas tqdm numpy

import os
import json
import random
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, precision_recall_curve
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm
import os

SEED = 42

MODEL_NAME = "distilbert-base-multilingual-cased"
TEXT_COL = "supplier_room_name"
TARGET_COL = "target"

TRAIN_PATH = "public_dataset.csv"
TEST_PATH = "new_submission_sample (3) (1).csv"

OUTPUT_DIR = "room_text_distilmbert_full"

MAX_LEN = 64
TRAIN_BATCH_SIZE = 16
VALID_BATCH_SIZE = 32
PRED_BATCH_SIZE = 64

LR = 2e-5
WEIGHT_DECAY = 0.01
TOTAL_EPOCHS = 9
VALID_SIZE = 0.15
MIN_PRECISION = 0.90
GRAD_CLIP = 1.0
NUM_WORKERS = os.cpu_count()

# =========================================================
# FULL TRAIN / DEBUG
# =========================================================
USE_DATA_FRACTION = False
DATA_FRACTION = 1.0
SKIP_SUBMISSION = False
# USE_DATA_FRACTION = True
# DATA_FRACTION = 0.03
# SKIP_SUBMISSION = False
# =========================================================
# RESUME
# =========================================================
# Для старта с нуля:
RESUME_CHECKPOINT = None

# Для продолжения обучения:
# RESUME_CHECKPOINT = os.path.join(OUTPUT_DIR, "last_checkpoint.pt")


In [4]:
# =========================================================
# PATHS
# =========================================================
from torch.optim import AdamW
os.makedirs(OUTPUT_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(OUTPUT_DIR, "best_model.pt")
LAST_CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "last_checkpoint.pt")
SPLIT_PATH = os.path.join(OUTPUT_DIR, "split_indices.npz")
CONFIG_PATH = os.path.join(OUTPUT_DIR, "train_config.json")
SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission.csv")


# =========================================================
# DEVICE / AMP
# =========================================================
USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")

USE_BF16 = USE_CUDA and torch.cuda.is_bf16_supported()
USE_FP16 = USE_CUDA and not USE_BF16
USE_AMP = USE_CUDA
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
USE_SCALER = USE_FP16


def amp_autocast():
    if not USE_AMP:
        return nullcontext()
    return torch.amp.autocast("cuda", dtype=AMP_DTYPE)


# =========================================================
# REPRODUCIBILITY
# =========================================================
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)


# =========================================================
# TEXT NORMALIZATION
# =========================================================
def normalize_text(x: str) -> str:
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\xa0", " ")
    x = x.replace("&amp;", "&")
    x = x.replace("“", '"').replace("”", '"')
    x = x.replace("’", "'")
    x = " ".join(x.split())
    return x.strip()


def sample_fraction_df(df: pd.DataFrame, frac: float, seed: int = 42) -> pd.DataFrame:
    if frac >= 1.0:
        return df.reset_index(drop=True)

    if frac <= 0:
        raise ValueError("frac must be > 0")

    n = max(1, int(len(df) * frac))
    return df.sample(n=n, random_state=seed, replace=False).reset_index(drop=True)


# =========================================================
# DATASET & COLLATOR
# =========================================================
class RoomTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len: int):
        self.texts = list(texts)
        self.labels = None if labels is None else list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding=False,
            return_tensors="pt",
        )

        item = {k: v.squeeze(0) for k, v in enc.items()}

        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item


class Collator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, batch):
        labels = None
        if "labels" in batch[0]:
            labels = torch.stack([x["labels"] for x in batch])

        features = [{k: v for k, v in x.items() if k != "labels"} for x in batch]
        padded = self.tokenizer.pad(
            features,
            padding=True,
            return_tensors="pt",
        )

        if labels is not None:
            padded["labels"] = labels

        return padded


# =========================================================
# METRICS
# =========================================================
def find_threshold_for_precision(y_true, y_prob, min_precision=0.90):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)

    precision = precision[:-1]
    recall = recall[:-1]

    mask = precision >= min_precision
    if not mask.any():
        return 1.0, 0.0, 0.0

    valid_thresholds = thresholds[mask]
    valid_precision = precision[mask]
    valid_recall = recall[mask]

    best_idx = np.argmax(valid_recall)

    return (
        float(valid_thresholds[best_idx]),
        float(valid_precision[best_idx]),
        float(valid_recall[best_idx]),
    )


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    losses = []
    all_probs = []
    all_labels = []

    for batch in tqdm(loader, desc="valid", leave=False):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        with amp_autocast():
            outputs = model(**batch)

        loss = outputs.loss
        probs = F.softmax(outputs.logits.float(), dim=1)[:, 1]

        losses.append(float(loss.item()))
        all_probs.append(probs.detach().float().cpu().numpy())
        all_labels.append(batch["labels"].detach().cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    pr_auc = average_precision_score(all_labels, all_probs)
    thr, prec, rec = find_threshold_for_precision(
        all_labels, all_probs, min_precision=MIN_PRECISION
    )

    return {
        "loss": float(np.mean(losses)),
        "pr_auc": float(pr_auc),
        "threshold_p90": thr,
        "precision_at_thr": prec,
        "recall_at_thr": rec,
        "probs": all_probs,
        "labels": all_labels,
    }


# =========================================================
# LLRD OPTIMIZER BUILDER
# =========================================================
def get_optimizer_grouped_parameters_llrd(model, lr=2e-5, lr_decay=0.9, weight_decay=0.01):
    base_model = getattr(model, 'bert', getattr(model, 'roberta', getattr(model, 'distilbert', None)))
    
    if base_model is None:
        raise ValueError("Не удалось определить базовую модель в `model`")

    no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]
    
    head_params = [
        (n, p) for n, p in model.named_parameters() 
        if not n.startswith(base_model.base_model_prefix)
    ]
    
    grouped_parameters = [
        {
            "params": [p for n, p in head_params if not any(nd in n for nd in no_decay)],
            "weight_decay": weight_decay,
            "lr": lr,
        },
        {
            "params": [p for n, p in head_params if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
            "lr": lr,
        },
    ]

    layers = getattr(base_model, "encoder", base_model).layer if hasattr(base_model, "encoder") else base_model.transformer.layer
    num_layers = len(layers)
    current_lr = lr * lr_decay

    for i in reversed(range(num_layers)):
        layer = layers[i]
        grouped_parameters.extend([
            {
                "params": [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)],
                "weight_decay": weight_decay,
                "lr": current_lr,
            },
            {
                "params": [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)],
                "weight_decay": 0.0,
                "lr": current_lr,
            },
        ])
        current_lr *= lr_decay

    embeddings = base_model.embeddings
    grouped_parameters.extend([
        {
            "params": [p for n, p in embeddings.named_parameters() if not any(nd in n for nd in no_decay)],
            "weight_decay": weight_decay,
            "lr": current_lr,
        },
        {
            "params": [p for n, p in embeddings.named_parameters() if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
            "lr": current_lr,
        },
    ])

    return grouped_parameters


# =========================================================
# SAVE / LOAD
# =========================================================
def save_json_config():
    config = {
        "seed": SEED,
        "model_name": MODEL_NAME,
        "text_col": TEXT_COL,
        "target_col": TARGET_COL,
        "max_len": MAX_LEN,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "valid_batch_size": VALID_BATCH_SIZE,
        "pred_batch_size": PRED_BATCH_SIZE,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "total_epochs": TOTAL_EPOCHS,
        "valid_size": VALID_SIZE,
        "min_precision": MIN_PRECISION,
        "grad_clip": GRAD_CLIP,
        "use_data_fraction": USE_DATA_FRACTION,
        "data_fraction": DATA_FRACTION,
    }
    with open(CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=2)


def save_split_indices(train_idx: np.ndarray, valid_idx: np.ndarray, path: str):
    np.savez(path, train_idx=train_idx, valid_idx=valid_idx)


def load_split_indices(path: str):
    data = np.load(path)
    return data["train_idx"], data["valid_idx"]


def save_best_model(model, tokenizer, best_pr_auc: float):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "model_name": MODEL_NAME,
            "max_len": MAX_LEN,
            "best_pr_auc": best_pr_auc,
        },
        BEST_MODEL_PATH,
    )
    tokenizer.save_pretrained(OUTPUT_DIR)


def save_last_checkpoint(model, optimizer, scaler, epoch: int, best_pr_auc: float):
    checkpoint = {
        "epoch": epoch,
        "best_pr_auc": best_pr_auc,
        "model_name": MODEL_NAME,
        "max_len": MAX_LEN,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "seed": SEED,
    }
    torch.save(checkpoint, LAST_CHECKPOINT_PATH)


def load_full_checkpoint(checkpoint_path: str, model, optimizer, scaler):
    ckpt = torch.load(checkpoint_path, map_location=DEVICE)

    model.load_state_dict(ckpt["model_state_dict"])
    model.to(DEVICE)
    model.float()

    optimizer.load_state_dict(ckpt["optimizer_state_dict"])

    if scaler is not None and ckpt.get("scaler_state_dict") is not None:
        scaler.load_state_dict(ckpt["scaler_state_dict"])

    start_epoch = ckpt["epoch"] + 1
    best_pr_auc = ckpt["best_pr_auc"]

    return start_epoch, best_pr_auc


def load_best_model_for_inference():
    ckpt = torch.load(BEST_MODEL_PATH, map_location=DEVICE)

    tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(
        ckpt["model_name"],
        num_labels=2,
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(DEVICE)
    model.float()
    model.eval()

    return tokenizer, model


# =========================================================
# DATALOADERS
# =========================================================
def build_dataloaders(tokenizer, train_part: pd.DataFrame, valid_part: pd.DataFrame):
    train_ds = RoomTextDataset(
        texts=train_part[TEXT_COL].tolist(),
        labels=train_part[TARGET_COL].tolist(),
        tokenizer=tokenizer,
        max_len=MAX_LEN,
    )

    valid_ds = RoomTextDataset(
        texts=valid_part[TEXT_COL].tolist(),
        labels=valid_part[TARGET_COL].tolist(),
        tokenizer=tokenizer,
        max_len=MAX_LEN,
    )

    collator = Collator(tokenizer)

    train_loader = DataLoader(
        train_ds,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=USE_CUDA,
        collate_fn=collator,
    )

    valid_loader = DataLoader(
        valid_ds,
        batch_size=VALID_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=USE_CUDA,
        collate_fn=collator,
    )

    return train_loader, valid_loader


# =========================================================
# PREDICT
# =========================================================
@torch.no_grad()
def predict_proba(texts, tokenizer, model):
    ds = RoomTextDataset(
        texts=[normalize_text(x) for x in texts],
        labels=None,
        tokenizer=tokenizer,
        max_len=MAX_LEN,
    )

    collator = Collator(tokenizer)

    loader = DataLoader(
        ds,
        batch_size=PRED_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=USE_CUDA,
        collate_fn=collator,
    )

    probs = []
    for batch in tqdm(loader, desc="predict", leave=False):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        with amp_autocast():
            logits = model(**batch).logits

        batch_probs = F.softmax(logits.float(), dim=1)[:, 1]
        probs.append(batch_probs.detach().float().cpu().numpy())

    return np.concatenate(probs)


# =========================================================
# TRAIN
# =========================================================
def train_model():
    save_json_config()

    df = pd.read_csv(TRAIN_PATH)
    df[TEXT_COL] = df[TEXT_COL].map(normalize_text)
    df[TEXT_COL] = df[TEXT_COL].fillna("")
    df[TARGET_COL] = df[TARGET_COL].astype(int)

    if os.path.exists(SPLIT_PATH):
        train_idx, valid_idx = load_split_indices(SPLIT_PATH)
        print(f"Loaded existing split: {SPLIT_PATH}")
    else:
        all_idx = np.arange(len(df))
        train_idx, valid_idx = train_test_split(
            all_idx,
            test_size=VALID_SIZE,
            random_state=SEED,
            stratify=df[TARGET_COL].values,
        )
        save_split_indices(train_idx, valid_idx, SPLIT_PATH)
        print(f"Saved split: {SPLIT_PATH}")

    train_part = df.iloc[train_idx].copy().reset_index(drop=True)
    valid_part = df.iloc[valid_idx].copy().reset_index(drop=True)

    if USE_DATA_FRACTION:
        train_part = sample_fraction_df(train_part, DATA_FRACTION, SEED)
        valid_part = sample_fraction_df(valid_part, DATA_FRACTION, SEED)
        print("FAST CHECK MODE")
        print(f"DATA_FRACTION = {DATA_FRACTION:.2%}")

    print("train:", train_part.shape, "valid:", valid_part.shape)
    print("target rate train:", train_part[TARGET_COL].mean())
    print("target rate valid:", valid_part[TARGET_COL].mean())

    if os.path.exists(os.path.join(OUTPUT_DIR, "tokenizer_config.json")):
        tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
    else:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
    )
    model.to(DEVICE)
    model.float()

    print("first param dtype:", next(model.parameters()).dtype)

    train_loader, valid_loader = build_dataloaders(tokenizer, train_part, valid_part)

    # Инициализируем LLRD Оптимизатор
    optimizer_grouped_parameters = get_optimizer_grouped_parameters_llrd(
        model, 
        lr=LR, 
        lr_decay=0.9, 
        weight_decay=WEIGHT_DECAY
    )
    optimizer = AdamW(optimizer_grouped_parameters)

    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER) if USE_CUDA else None

    start_epoch = 1
    best_pr_auc = -1.0

    if RESUME_CHECKPOINT is not None:
        if not os.path.exists(RESUME_CHECKPOINT):
            raise FileNotFoundError(f"Checkpoint not found: {RESUME_CHECKPOINT}")

        print(f"Resuming from: {RESUME_CHECKPOINT}")
        start_epoch, best_pr_auc = load_full_checkpoint(
            checkpoint_path=RESUME_CHECKPOINT,
            model=model,
            optimizer=optimizer,
            scaler=scaler,
        )
        print(f"Resume start_epoch={start_epoch}, best_pr_auc={best_pr_auc:.6f}")
        print("first param dtype after resume:", next(model.parameters()).dtype)

    for epoch in range(start_epoch, TOTAL_EPOCHS + 1):
        model.train()
        train_losses = []

        progress = tqdm(train_loader, desc=f"epoch {epoch}/{TOTAL_EPOCHS}")
        for batch in progress:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)

            with amp_autocast():
                outputs = model(**batch)
                loss = outputs.loss

            if USE_SCALER:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()

            train_losses.append(float(loss.item()))
            progress.set_postfix(loss=f"{np.mean(train_losses):.4f}")

        valid_metrics = evaluate(model, valid_loader)

        print(
            f"\nEpoch {epoch}: "
            f"train_loss={np.mean(train_losses):.4f} | "
            f"valid_loss={valid_metrics['loss']:.4f} | "
            f"valid_PR_AUC={valid_metrics['pr_auc']:.6f} | "
            f"thr@P>={MIN_PRECISION:.2f}={valid_metrics['threshold_p90']:.4f} | "
            f"precision={valid_metrics['precision_at_thr']:.4f} | "
            f"recall={valid_metrics['recall_at_thr']:.4f}"
        )

        save_last_checkpoint(
            model=model,
            optimizer=optimizer,
            scaler=scaler,
            epoch=epoch,
            best_pr_auc=max(best_pr_auc, valid_metrics["pr_auc"]),
        )
        print(f"Saved last checkpoint: {LAST_CHECKPOINT_PATH}")

        if valid_metrics["pr_auc"] > best_pr_auc:
            best_pr_auc = valid_metrics["pr_auc"]
            save_best_model(model, tokenizer, best_pr_auc)
            print(f"Saved best model: {BEST_MODEL_PATH}")


# =========================================================
# SUBMISSION
# =========================================================
def build_submission():
    if not os.path.exists(BEST_MODEL_PATH):
        raise FileNotFoundError(f"Best model not found: {BEST_MODEL_PATH}")

    tokenizer, model = load_best_model_for_inference()

    test_df = pd.read_csv(TEST_PATH)
    test_df[TEXT_COL] = test_df[TEXT_COL].map(normalize_text)
    test_df[TEXT_COL] = test_df[TEXT_COL].fillna("")

    probs = predict_proba(test_df[TEXT_COL].tolist(), tokenizer, model)

    if "Unnamed: 0" in test_df.columns:
        row_id = test_df["Unnamed: 0"]
    elif "row_id" in test_df.columns:
        row_id = test_df["row_id"]
    else:
        row_id = np.arange(len(test_df))

    submission = pd.DataFrame({
        "row_id": row_id,
        "target": probs,
    })

    submission.to_csv(SUBMISSION_PATH, index=False)
    np.save(os.path.join(OUTPUT_DIR, "test_bert_probs.npy"), probs)

    print("\nSubmission preview:")
    print(submission.head())
    print(f"Saved submission: {SUBMISSION_PATH}")


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    print("DEVICE:", DEVICE)
    print("MODEL_NAME:", MODEL_NAME)
    print("USE_AMP:", USE_AMP)
    print("USE_BF16:", USE_BF16)
    print("USE_FP16:", USE_FP16)
    print("USE_SCALER:", USE_SCALER)
    print("RESUME_CHECKPOINT:", RESUME_CHECKPOINT)
    print("USE_DATA_FRACTION:", USE_DATA_FRACTION)
    print("DATA_FRACTION:", DATA_FRACTION)

    train_model()

    if not SKIP_SUBMISSION:
        build_submission()

DEVICE: cuda
MODEL_NAME: distilbert-base-multilingual-cased
USE_AMP: True
USE_BF16: True
USE_FP16: False
USE_SCALER: False
RESUME_CHECKPOINT: None
USE_DATA_FRACTION: False
DATA_FRACTION: 1.0
Loaded existing split: room_text_distilmbert_full/split_indices.npz
train: (156517, 3) valid: (27621, 3)
target rate train: 0.5159311768050755
target rate valid: 0.5159480105716665


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


first param dtype: torch.float32


epoch 1/9:   0%|          | 0/9783 [00:00<?, ?it/s]

...


Epoch 9: train_loss=0.2205 | valid_loss=0.3704 | valid_PR_AUC=0.940919 | thr@P>=0.90=0.7708 | precision=0.9000 | recall=0.8383
Saved last checkpoint: room_text_distilmbert_full/last_checkpoint.pt


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


predict:   0%|          | 0/172 [00:00<?, ?it/s]


Submission preview:
   row_id    target
0       0  0.001782
1       1  0.286968
2       2  0.988492
3       3  0.993203
4       4  0.000830
Saved submission: room_text_distilmbert_full/submission.csv
